# 03 Generate baseline summaries
The fine-tuned model writes one TLDR for each of the 618 test abstracts, with the one decoding method set in `configs/config.yaml`.

Reads `outputs/model/` and `data/processed/test.jsonl`. Writes `outputs/baseline.csv`.
**The baseline is frozen once written:** do not change the model or the decoding settings after seeing any scores.

In [1]:
import json, os
from pathlib import Path

import pandas as pd
import torch, yaml
from IPython.display import display
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

if Path.cwd().name == "notebooks":
    os.chdir("..")                      # run everything from the repo root
ROOT = Path.cwd()
CFG = yaml.safe_load((ROOT / "configs" / "config.yaml").read_text())
D, G, BATCH = CFG["data"], CFG["baseline"]["decoding"], CFG["baseline"]["batch_size"]
MODEL_DIR, BASELINE = ROOT / "outputs" / "model", ROOT / "outputs" / "baseline.csv"
OVERWRITE = False                       # True = write baseline.csv again (only if nothing has been scored yet)
pd.set_option("display.max_colwidth", 200)

test = pd.read_json(ROOT / "data" / "processed" / "test.jsonl", orient="records", lines=True, dtype=False)
assert len(test) == D["sizes"]["test"], "run 01_data first"

device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).to(device).eval()

# The model folder also carries default generation settings. This shows what is actually used:
# the config values below override them, anything else comes from the model folder.
effective = {**model.generation_config.to_dict(), **G}
keys = ["num_beams", "do_sample", "max_new_tokens", "no_repeat_ngram_size", "repetition_penalty",
        "length_penalty", "min_length", "early_stopping", "forced_bos_token_id"]
display(pd.DataFrame({"set in config": {k: G.get(k, "") for k in keys}, "effective": {k: effective.get(k) for k in keys}}))

,set in config,effective
num_beams,1,1
do_sample,False,False
max_new_tokens,64,64
no_repeat_ngram_size,3,3
repetition_penalty,,1.0
length_penalty,,1.0
min_length,,0
early_stopping,,True
forced_bos_token_id,,0


## Generate

In [2]:
def generate():
    summaries = []
    for i in tqdm(range(0, len(test), BATCH), desc="generating"):
        batch = tok(test.abstract.iloc[i:i + BATCH].tolist(), max_length=D["max_source_len"],
                    truncation=True, padding=True, return_tensors="pt").to(device)
        with torch.inference_mode():
            ids = model.generate(**batch, **G)
        summaries += [s.strip() for s in tok.batch_decode(ids, skip_special_tokens=True)]
    pd.DataFrame({"id": test.id, "abstract": test.abstract, "summary": summaries,
                  "human_tldrs": test.human_tldrs.map(lambda t: json.dumps(t, ensure_ascii=False))}
                 ).to_csv(BASELINE, index=False, lineterminator="\n")
    print("wrote", BASELINE)

if BASELINE.exists() and not OVERWRITE:
    print("baseline.csv already exists and is frozen, so generation is skipped. Set OVERWRITE = True to write it again.")
else:
    generate()

generating:   0%|          | 0/39 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


wrote c:\Users\SAM\Documents\Repositories\csc173-midterm\outputs\baseline.csv


## Validate and look at the output

In [3]:
b = pd.read_csv(BASELINE, keep_default_na=False)
checks = {
    "one summary per test abstract (618 rows)": len(b) == D["sizes"]["test"],
    "ids match the test split, same order": list(b["id"]) == list(test["id"]),
    "abstracts match the test split": bool((b["abstract"].values == test["abstract"].values).all()),
    "no empty summaries": bool((b["summary"].str.strip() != "").all()),
}
display(pd.Series(checks, name="passed").to_frame())
assert all(checks.values()), "validation failed"

b["human_first"] = b.human_tldrs.map(lambda s: json.loads(s)[0])
display(b.head(5)[["id", "summary", "human_first"]])
words = pd.DataFrame({"summary": b.summary.str.split().str.len(), "human": b.human_first.str.split().str.len()})
display(words.describe().round(1).T)
copied = b.apply(lambda r: r.summary.lower() in r.abstract.lower(), axis=1)
cut = ~b.summary.str.endswith((".", "?", "!"))
print(f"copied verbatim from the abstract: {copied.sum()} / {len(b)}")
print(f"not ending in . ? or ! (possibly cut off): {cut.sum()} / {len(b)}")

,passed
one summary per test abstract (618 rows),True
"ids match the test split, same order",True
abstracts match the test split,True
no empty summaries,True


,id,summary,human_first
0,SJ1Xmf-Rb,"We propose FearNet, a generative model that does not store previous examples, making it memory efficient.","FearNet is a memory efficient neural-network, inspired by memory formation in the mammalian brain, that is capable of incremental class learning without catastrophic forgetting."
1,S1xzyhR9Y7,Multi-view learning can provide self-supervision when different views are available of the same data.,Multi-view learning improves unsupervised sentence representation learning
2,HJDUjKeA-,"We show how discrete objects can be learnt in an unsupervised fashion from pixels, and how to perform reinforcement learning using this object representation.","We show how discrete objects can be learnt in an unsupervised fashion from pixels, and how to perform reinforcement learning using this object representation."
3,BJgLg3R9KQ,We extend a state-of-the-art attention network and demonstrate its benefit of using stronger supervisory signals by teaching DCNs to attend to image regions that humans deem important for object r...,"A large-scale dataset for training attention models for object recognition leads to more accurate, interpretable, and human-like object recognition."
4,BklpOo09tQ,"We show that adversarial examples can be generated ata cost comparable to that of two runs of the fast gradient sign method (FGSM), allowing defense against adversarial attacks with a robustness l...",We proposed a time-efficient defense method against one-step and iterative adversarial attacks.


,count,mean,std,min,25%,50%,75%,max
summary,618.0,21.1,7.5,5.0,15.0,20.0,25.0,52.0
human,618.0,18.9,7.6,3.0,13.0,18.0,24.0,44.0


copied verbatim from the abstract: 249 / 618
not ending in . ? or ! (possibly cut off): 19 / 618
